1. ffmpeg 설치
2. ffmpeg -i files/JENNIE_interview.mp4 -vn files/audio.mp3 커맨드 실행
- mp4 동영상 파일을 가져와서 영상은 무시하고 음성만 mp3로 추출해 저장하라는 의미
3. whisper에 사용하기 위해서 10분 단위로 자른다

In [ ]:
import subprocess ## 파이썬에서 커맨드를 실행 할 수 있도록 해주는 라이브러리
from pydub import AudioSegment
import math

def extract_audio_from_video(video_path, audio_path):
    command = ["ffmpeg", "-i", video_path, "-vn", audio_path]
    subprocess.run(command)
    
# extract_audio_from_video("./files/JENNIE_interview.mp4", "./files/audio.mp3")

def cut_audio_in_chunks(audio_path, chunk_size, chunks_folder):
    track = AudioSegment.from_mp3("./files/audio.mp3")
    # track.duration_seconds
    # ten_minutes = 10 * 60 * 1000 ## 밀리초 기준
    chunk_len = chunk_size * 60 *1000
    
    chunks = math.ceil(len(track) / chunk_len)
    for i in range(chunks):
        start_time = i * chunk_len
        end_time = (i+1) * chunk_len
        
        chunk = track[start_time:end_time]
        chunk.export(f"{chunks_folder}/chunk_{i}.mp3", format="mp3")

# cut_audio_in_chunks("./files/audio.mp3", 10, "./files/chunks")

In [17]:
import openai
import glob

# transcript = openai.Audio.transcribe("whisper-1", open("./files/chunks/chunk_0.mp3", "rb"))
# transcript

def trascribe_chunks(chunk_folder, destination):
    files = glob.glob(f"{chunk_folder}/*.mp3")
    for file in files:
        with open(file, "rb") as audio_file, open(destination, "a") as text_file:
            transcript = openai.Audio.transcribe("whisper-1", audio_file)
            text_file.write(transcript.text)

trascribe_chunks("./files/chunks", "./files/transcript.txt")

    